<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# MobileNetV1 on ImageNet - Akida 2 Hardware Benchmark

<p align="right">Run Time: ~10 minutes</p>

Measures what a MobileNetV1 ImageNet model actually costs on Akida 2 hardware: inference
latency, the mapping onto Neural Processors (NPs), and where the time goes layer by layer.

The Akida 2 reference platform is an FPGA running at 25 MHz. Because that clock is low, a
latency **projected** to the 1 GHz AKD2500 target clock is reported as well. The cycle count
does not depend on the clock, so the projection is an exact rescale. Power measurement is
not available on the FPGA yet, so this benchmark is latency-only.

This notebook needs no dataset setup - it runs from a 10-image sample pack fetched
automatically.

> **Requires a connected Akida 2 device.** Without one, the hardware sections report "no
> hardware found" and are skipped. This will not run on Colab.

## Configuration

Choose the model by width multiplier (1.0 or 1.25) and input resolution (224 or 256).

In [ ]:
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ
from imagenet_mobilenet_model import TIMM_NAMES

ALPHA = 1.0          # 1.0 or 1.25
RESOLUTION = 224     # 224 or 256
NUM_SAMPLES = 100    # inferences per benchmark

MEASURED_CLOCK = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']  # 25 MHz FPGA
PROJECTED_CLOCK = AKIDA_CLOCKS_HZ['AKD2500']     # 1 GHz target (pre-production)

MODEL_NAME = f'{TIMM_NAMES[ALPHA]} (alpha={ALPHA}, {RESOLUTION}x{RESOLUTION})'
print(f'Model: {MODEL_NAME}')

## Model

Load the converted Akida model from `pretrained_models/`. These files are tracked with Git
LFS - if you get a parse error here, run `git lfs pull`.

In [ ]:
from imagenet_mobilenet_model import load_model

akida_model = load_model(ALPHA, 'akida', resolution=RESOLUTION)
akida_model.summary()

## Samples

Akida is event-driven: a zero activation produces no event and therefore no work. Latency
depends on how sparse the activations are, which depends on the input. So the benchmark
must run on real images - random noise gives the wrong activity statistics and therefore
the wrong numbers.

Ten real images, cycled, capture those statistics well enough. They are cropped
for the chosen width and resolution, because each pair uses its own crop ratio. Pass `data_path` to
`get_samples` to draw from the ImageNet validation set instead, if you have it set up. The
samples are raw uint8 and channels-first, as the Akida model expects.

In [ ]:
from imagenet_mobilenet_data import get_samples

samples = get_samples(ALPHA, num_samples=NUM_SAMPLES, resolution=RESOLUTION)
print(f'{samples.shape} {samples.dtype}, range [{samples.min()}, {samples.max()}]')

## Hardware device detection

`get_akida_device` checks for a device matching the model's IP version, so an Akida 2 model
does not get mapped onto Akida 1 silicon.

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=akida_model.ip_version)
if device is not None:
    print(f'Akida hardware found: {device}')
else:
    print('No hardware found - the benchmark sections below will be skipped.')

## Activation sparsity

Worth looking at before the timings, because it predicts them. Layers with high input
sparsity are the ones the hardware gets through quickly. This needs no hardware.

Every layer is listed, because the per-layer plot below needs each layer's input. In
MobileNetV1 every convolution ends in a ReLU6; only the classifier has no activation, so
the headline mean is over the ReLU layers only.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

from imagenet_mobilenet_eval import mean_activation_sparsity

sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)
mean_sparsity = mean_activation_sparsity(akida_model, sparsity_dict)
print(f'\nMean activation sparsity (ReLU layers): {mean_sparsity * 100:.2f}%')

## Full model benchmark

Three mapping modes:

- **Minimal** packs the model onto the fewest NPs it fits in.
- **AllNps** spreads it across more NPs while keeping the number of hardware passes down.
- **HwPr** also uses every NP it can, but lets the work spread over more passes.

Each is measured at the FPGA clock, then projected to the target clock from the cycle
count.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

full_results = {}
if device is not None:
    for mm in ['Minimal', 'AllNps', 'HwPr']:
        map_mode = getattr(akida.MapMode, mm)
        print(f'Running full-model benchmark (MapMode={mm})...')
        res = full_model_benchmark(akida_model, device, samples,
                                   map_mode=map_mode, clock_freq=MEASURED_CLOCK)
        if res is None:
            # Not every mode maps every model onto a single hardware sequence
            print(f'  MapMode={mm} did not map to hardware - skipping this mode.')
            continue
        res['projected_clk_ms'] = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        full_results[mm] = res
        # Re-map without hw_only so akida_model.sequences is populated for the stats
        akida_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(akida_model)
        full_results[mm]['num_nps'] = num_nps
        full_results[mm]['num_passes'] = num_passes
        print(f'  {num_nps} NP(s), {num_passes} pass(es), {num_sequences} sequence(s), '
              f'{res["mean_clk_ms"]:.3f} ms @ {MEASURED_CLOCK / 1e6:.0f} MHz, '
              f'{res["projected_clk_ms"]:.3f} ms @ {PROJECTED_CLOCK / 1e6:.0f} MHz')
        if num_sequences > 1:
            print('  WARNING: model not completely mapped to hardware')
else:
    print('No hardware - skipping.')

In [ ]:
from brainchip_utils.plot_utils import plot_full_model_results

if device is not None:
    plot_full_model_results(full_results, akida_model, device, model_name=MODEL_NAME)

## Per-layer benchmark

Full-model timing tells us the total cost but not where time is spent.
`per_layer_benchmark` from
[brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py)
reconstructs latency layer by layer by running cumulative sub-models and differencing the
results.

A layer's processing time depends on the volume of its inputs, its number of filters, and
the number of NPs allocated to it. Because Akida only processes non-zero activations, it
also scales with the layer's *input* sparsity, so the per-layer timings and the sparsity
values above are naturally correlated.

In [ ]:
from brainchip_utils.hardware_utils import per_layer_benchmark

if device is not None:
    akida_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    print(f'Running per-layer benchmark ({NUM_SAMPLES} samples)...')
    per_layer_results = per_layer_benchmark(akida_model, device, samples,
                                            repeats=NUM_SAMPLES, clock_freq=MEASURED_CLOCK)
else:
    print('No hardware - skipping.')

In [ ]:
from brainchip_utils.plot_utils import plot_per_layer_results

if device is not None:
    # Map without hw_only so the mapping plot can be drawn
    akida_model.map(device, mode=akida.MapMode.Minimal)
    plot_per_layer_results(per_layer_results, akida_model, sparsity_dict,
                           model_name=MODEL_NAME)

## Summary

In [ ]:
print(MODEL_NAME)
print(f'  mean activation sparsity (ReLU layers): {mean_sparsity * 100:.2f}%')
if device is not None:
    for mm, res in full_results.items():
        print(f'  {mm:8} {res["num_nps"]:3d} NPs, {res["num_passes"]:2d} pass(es), '
              f'{res["mean_clk_ms"]:8.3f} ms @ {MEASURED_CLOCK / 1e6:.0f} MHz, '
              f'{res["projected_clk_ms"]:7.3f} ms @ {PROJECTED_CLOCK / 1e6:.0f} MHz (projected)')
else:
    print('  no hardware present - latency not measured')